# Statistical Process Control (SPC) Anomaly Detection

This notebook demonstrates how to use SPC methods for detecting anomalies in electricity load data.

## Overview

Statistical Process Control includes several methods:
1. **Shewhart Charts** - Individual points beyond control limits
2. **EWMA (Exponentially Weighted Moving Average)** - Detects small shifts over time
3. **CUSUM (Cumulative Sum Control Chart)** - Accumulates deviations to detect sustained shifts
4. **Nelson Rules** - Pattern-based detection rules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import our SPC module
from spc_anomaly_detection import SPCDetector, SPCConfig, MultiSeriesSPCDetector, visualize_spc_results

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Generate Synthetic Data Example

First, let's create synthetic data to demonstrate SPC methods.

In [ ]:
# Generate synthetic time series with anomalies
np.random.seed(42)
n_points = 500

# Base signal with trend
t = np.arange(n_points)
base_signal = 100 + 0.1 * t + np.random.normal(0, 5, n_points)

# Add anomalies
data = base_signal.copy()
data[100:110] += 30  # Step shift (10 consecutive high values)
data[200] = 150      # Spike
data[250:265] -= 20  # Negative shift
data[350] = 50       # Extreme low value

plt.figure(figsize=(14, 4))
plt.plot(data, label='Data with anomalies', linewidth=1.5)
plt.axvspan(100, 110, alpha=0.2, color='red', label='Anomaly: Step shift')
plt.axvline(200, color='red', linestyle='--', alpha=0.5, label='Anomaly: Spike')
plt.axvspan(250, 265, alpha=0.2, color='orange', label='Anomaly: Negative shift')
plt.axvline(350, color='purple', linestyle='--', alpha=0.5, label='Anomaly: Extreme')
plt.xlabel('Time Index')
plt.ylabel('Consumption')
plt.title('Synthetic Data with Anomalies')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Data shape: {data.shape}")
print(f"Mean: {np.mean(data):.2f}, Std: {np.std(data):.2f}")
print(f"Min: {np.min(data):.2f}, Max: {np.max(data):.2f}")

## 2. Initialize and Fit SPC Detector

We'll use the first half of data (assumed clean) to establish baseline statistics.

In [ ]:
# Create SPC configuration
config = SPCConfig(
    shewhart_multiplier=3.0,      # 3-sigma limits
    ewma_lambda=0.2,              # Smoothing parameter (typical range: 0.05-0.3)
    ewma_multiplier=2.66,         # Equivalent to ~3-sigma for EWMA
    cusum_k_value=0.5,            # CUSUM reference value
    cusum_h_value=4.77,           # CUSUM decision interval
    enable_nelson_rules=True,
    min_baseline_points=30,
)

print("SPC Configuration:")
print(f"  Shewhart multiplier: {config.shewhart_multiplier} sigma")
print(f"  EWMA lambda: {config.ewma_lambda}")
print(f"  EWMA multiplier: {config.ewma_multiplier} sigma")
print(f"  CUSUM k: {config.cusum_k_value} sigma, h: {config.cusum_h_value} sigma")

# Create detector and fit baseline
detector = SPCDetector(config)
baseline_data = data[:250]  # Use first half as baseline (assumed clean)

detector.fit(baseline_data)

print(f"\nBaseline statistics:")
print(f"  Mean: {detector.baseline_mean:.2f}")
print(f"  Std Dev: {detector.baseline_std:.2f}")
print(f"  Baseline size: {detector.baseline_size}")
print(f"  Control limits (±3σ): [{detector.baseline_mean - 3*detector.baseline_std:.2f}, {detector.baseline_mean + 3*detector.baseline_std:.2f}]")

## 3. Individual Control Charts (Shewhart Chart)

The Shewhart or I-chart plots individual values against control limits.

In [ ]:
# Get Shewhart chart components
vals, center, ucl, lcl = detector.shewhart_chart(data)

# Plot
plt.figure(figsize=(14, 6))
plt.plot(data, 'b-', label='Data', alpha=0.7, linewidth=1)
plt.axhline(center, color='g', linestyle='--', linewidth=2, label='Center Line (Mean)')
plt.axhline(ucl[0], color='r', linestyle='--', linewidth=2, label='UCL (Upper Control Limit)')
plt.axhline(lcl[0], color='r', linestyle='--', linewidth=2, label='LCL (Lower Control Limit)')
plt.fill_between(range(len(data)), lcl, ucl, alpha=0.1, color='gray', label='In-Control Zone')

# Highlight points outside control limits
out_of_control = (vals > ucl) | (vals < lcl)
plt.scatter(np.where(out_of_control)[0], data[out_of_control], 
           color='red', s=100, marker='X', label='Out of Control', zorder=5)

plt.xlabel('Time Index')
plt.ylabel('Consumption')
plt.title('Shewhart Control Chart (I-Chart)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Points outside control limits: {out_of_control.sum()}")
print(f"Out-of-control indices: {np.where(out_of_control)[0].tolist()}")

## 4. EWMA (Exponentially Weighted Moving Average)

EWMA gives more weight to recent observations and is effective at detecting small shifts.

In [ ]:
# Get EWMA chart components
ewma_vals, center, ucl, lcl = detector.ewma_chart(data)

# Plot
plt.figure(figsize=(14, 6))
plt.plot(data, 'b-', label='Original Data', alpha=0.5, linewidth=1)
plt.plot(ewma_vals, 'purple', linewidth=2.5, label=f'EWMA (λ={config.ewma_lambda})')
plt.axhline(center, color='g', linestyle='--', linewidth=2, label='Center Line')
plt.fill_between(range(len(data)), lcl, ucl, alpha=0.15, color='gray', label='Control Limits')

# Highlight out-of-control points
out_of_control_ewma = (ewma_vals > ucl) | (ewma_vals < lcl)
plt.scatter(np.where(out_of_control_ewma)[0], ewma_vals[out_of_control_ewma],
           color='red', s=100, marker='X', label='Out of Control', zorder=5)

plt.xlabel('Time Index')
plt.ylabel('Consumption')
plt.title('EWMA Control Chart')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Points outside EWMA control limits: {out_of_control_ewma.sum()}")
print(f"Out-of-control indices: {np.where(out_of_control_ewma)[0].tolist()}")

## 5. CUSUM (Cumulative Sum Control Chart)

CUSUM is sensitive to small systematic shifts in the process mean.

In [ ]:
# Get CUSUM chart components
cusum_vals, center, decision_limit, _ = detector.cusum_chart(data)

# Plot
plt.figure(figsize=(14, 6))
colors = ['red' if v > decision_limit[0] else 'steelblue' for v in cusum_vals]
plt.bar(range(len(data)), cusum_vals, color=colors, alpha=0.7, label='CUSUM')
plt.axhline(decision_limit[0], color='red', linestyle='--', linewidth=2, 
           label=f'Decision Limit (h={config.cusum_h_value}σ)')
plt.axhline(0, color='black', linestyle='-', linewidth=0.5)

plt.xlabel('Time Index')
plt.ylabel('CUSUM Value')
plt.title('CUSUM Control Chart (Detects Sustained Shifts)')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.show()

print(f"Times CUSUM exceeded decision limit: {(cusum_vals > decision_limit[0]).sum()}")
print(f"Max CUSUM value: {np.max(cusum_vals):.2f}")
print(f"Decision limit: {decision_limit[0]:.2f}")

## 6. Nelson Rules

Nelson Rules are pattern-based rules for detecting out-of-control conditions.

In [ ]:
# Detect anomalies using Nelson Rules
nelson_anomalies = detector.nelson_rules(data)

# Plot
plt.figure(figsize=(14, 6))
plt.plot(data, 'b-', label='Data', alpha=0.7, linewidth=1)
plt.axhline(detector.baseline_mean, color='g', linestyle='--', label='Center', linewidth=1.5)

# Add sigma bands
for sigma in [1, 2, 3]:
    plt.axhline(detector.baseline_mean + sigma * detector.baseline_std, 
               color='gray', linestyle=':', alpha=0.3, linewidth=0.8)
    plt.axhline(detector.baseline_mean - sigma * detector.baseline_std,
               color='gray', linestyle=':', alpha=0.3, linewidth=0.8)

# Highlight Nelson rule violations
plt.scatter(np.where(nelson_anomalies)[0], data[nelson_anomalies],
           color='red', s=100, marker='X', label='Nelson Rule Violation', zorder=5)

plt.xlabel('Time Index')
plt.ylabel('Consumption')
plt.title('Nelson Rules Anomaly Detection')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Nelson rule violations detected: {nelson_anomalies.sum()}")
print(f"Violation indices: {np.where(nelson_anomalies)[0].tolist()}")

## 7. Combined Anomaly Detection

Use multiple methods together for more robust detection.

In [ ]:
# Run detection with all methods
anom_flags, results = detector.detect(
    data,
    methods=("shewhart", "ewma", "cusum", "nelson"),
    aggregate="any"  # Flag if ANY method detects anomaly
)

print("Anomaly Detection Results:")
print(f"Total anomalies detected (any method): {anom_flags.sum()}")
print(f"\nBreakdown by method:")
for col in results.columns:
    if 'anomaly' in col:
        print(f"  {col}: {results[col].sum()}")

# Show detailed results for anomalous points
anomaly_rows = results[anom_flags].copy()
anomaly_rows['index'] = anomaly_rows.index
print(f"\nTop 10 anomalies by index:")
print(anomaly_rows[['index', 'value', 'shewhart_anomaly', 'ewma_anomaly', 'cusum_anomaly', 'nelson_anomaly']].head(10))

## 8. Visualize All Methods Together

In [ ]:
visualize_spc_results(data, detector, title="Comprehensive SPC Analysis")

## 9. Apply to Real Data (if available)

Now let's apply SPC to your actual electricity data. Adjust the column names as needed.

In [ ]:
# Try to load actual data
from pathlib import Path

project_root = Path('.').resolve().parent
data_path = project_root / 'data' / 'train.csv'

if data_path.exists():
    print(f"Loading data from {data_path}")
    df = pd.read_csv(data_path)
    print(f"Data shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(df.head())
else:
    print(f"Data file not found at {data_path}")
    print("Adjust the path as needed.")
    df = None

## 10. Process Real Data

If you have multiple transformers/regions, process each separately.

In [ ]:
if df is not None:
    # Identify the transformer ID column and value column
    # Adjust these based on your actual column names
    id_columns = [c for c in df.columns if 'id' in c.lower() or 'transformer' in c.lower()]
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    
    print(f"Potential ID columns: {id_columns}")
    print(f"Numeric columns: {numeric_columns}")
    
    if id_columns and numeric_columns:
        id_col = id_columns[0]
        value_col = numeric_columns[0]
        
        print(f"\nUsing ID column: {id_col}")
        print(f"Using value column: {value_col}")
        
        # Process first transformer as example
        first_id = df[id_col].iloc[0]
        transformer_data = df[df[id_col] == first_id][value_col].values
        
        print(f"\nProcessing transformer {first_id}")
        print(f"Data length: {len(transformer_data)}")
        print(f"Mean: {np.mean(transformer_data):.2f}, Std: {np.std(transformer_data):.2f}")
        
        # Fit and detect
        detector_real = SPCDetector(config)
        baseline_size = max(30, len(transformer_data) // 2)
        detector_real.fit(transformer_data[:baseline_size])
        
        anom_real, results_real = detector_real.detect(
            transformer_data,
            methods=("shewhart", "ewma", "cusum", "nelson"),
            aggregate="any"
        )
        
        print(f"Anomalies detected: {anom_real.sum()} out of {len(transformer_data)}")
        print(f"Anomaly rate: {anom_real.mean()*100:.2f}%")
        print(f"\nAnomaly indices: {np.where(anom_real)[0].tolist()[:20]}...")
    else:
        print("Could not identify ID or value columns")
        print("Please manually specify the column names.")
else:
    print("Skipping real data processing (file not found)")